# Notebook 1 — Exploratory Data Analysis

Loads the synthetic 429-record CI/CD dataset and reproduces the paper's Figure 2 and Figure 3.

In [ ]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from src.data_generator import generate_dataset, save_dataset

In [ ]:
df = generate_dataset(seed=42)
save_dataset(df)
print(f'Dataset shape: {df.shape}')
df.head()

In [ ]:
print('=== Basic Statistics ===')
print(df[['build_duration','test_execution_time','deployment_frequency','resource_utilization','error_count']].describe().round(2))

In [ ]:
print(f"Total anomalies: {df['is_anomaly'].sum()} / {len(df)} ({df['is_anomaly'].mean()*100:.1f}%)")
print(df['anomaly_type'].value_counts())

In [ ]:
# Paper Figure 2: Build duration distribution (scatter with anomalies highlighted)
fig, ax = plt.subplots(figsize=(14, 5))
normal  = df[df['is_anomaly'] == 0]
anomaly = df[df['is_anomaly'] == 1]

ax.scatter(normal['instance_id'],  normal['build_duration'],  c='steelblue', alpha=0.6, s=18, label='Normal')
ax.scatter(anomaly['instance_id'], anomaly['build_duration'], c='red',       alpha=0.9, s=40, marker='x', label='Anomaly')
ax.axvspan(200, 300, alpha=0.07, color='red', label='Infrastructure instability band')
ax.set(xlabel='Pipeline Instance', ylabel='Build Duration (s)', title='Fig 2 — Build Duration with Detected Anomalies')
ax.legend()
plt.tight_layout()
plt.savefig('../results/nb_fig2_build_duration.png', dpi=150)
plt.show()

In [ ]:
# Paper Figure 3: Test execution time histogram
fig, ax1 = plt.subplots(figsize=(10, 5))
ax1.hist(df['test_execution_time'], bins=30, color='steelblue', alpha=0.7, label='Test Exec Time')
ax1.set(xlabel='Test Execution Time (s)', ylabel='Frequency', title='Fig 3 — Test Execution Time Distribution')

ax2 = ax1.twinx()
bucket = pd.cut(df['test_execution_time'], bins=30)
success_rate = df.groupby(bucket, observed=True)['is_anomaly'].apply(lambda x: 1 - x.mean())
ax2.plot(success_rate.values, color='orange', linewidth=2, label='Deployment Success Rate')
ax2.set_ylabel('Deployment Success Rate')
ax2.set_ylim(0, 1.2)

plt.tight_layout()
plt.savefig('../results/nb_fig3_test_hist.png', dpi=150)
plt.show()

In [ ]:
# Correlation heatmap
features = ['build_duration','test_execution_time','deployment_frequency','resource_utilization','error_count']
fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(df[features].corr(), annot=True, fmt='.2f', cmap='coolwarm', ax=ax)
ax.set_title('Feature Correlation Matrix')
plt.tight_layout()
plt.show()